# CSET 346 – Natural Language Processing: Project Assignment
## Detecting Stance in Media on Global Warming for Indic Languages (Hindi & Bengali)
**Claim:** *"Climate change and global warming is a serious concern."*
**Target Classes:** `Favour`, `Against`, `None`
**Instructor:** Dr. Supriya Chanda, Assistant Professor, SCSET, Bennett University

### 1. Install Dependencies

In [ ]:
!pip install -q transformers datasets torch scikit-learn seaborn matplotlib pandas numpy joblib

### 2. Verify GPU Acceleration

In [ ]:
import torch
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))

### 3. Load & Preprocess Data

In [ ]:
import pandas as pd
import unicodedata, re

def clean_indic_text(text):
    text = unicodedata.normalize('NFC', str(text))
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[!?.।]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Load datasets (keep_default_na=False avoids parsing 'None' class as NaN)
train_df = pd.read_csv('train.csv', keep_default_na=False)
val_df = pd.read_csv('val.csv', keep_default_na=False)
test_df = pd.read_csv('test.csv', keep_default_na=False)

train_df['cleaned'] = train_df['text'].apply(clean_indic_text)
val_df['cleaned'] = val_df['text'].apply(clean_indic_text)
test_df['cleaned'] = test_df['text'].apply(clean_indic_text)
print("Train samples:", len(train_df), "Val samples:", len(val_df), "Test samples:", len(test_df))

### 4. Classical ML Baseline (Word + Char N-Gram TF-IDF)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

vectorizer = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1,2), max_features=10000)),
    ('char', TfidfVectorizer(analyzer='char_wb', ngram_range=(2,4), max_features=15000))
])

X_train = vectorizer.fit_transform(train_df['cleaned'])
X_test = vectorizer.transform(test_df['cleaned'])

clf = LogisticRegression(C=2.0, max_iter=1000, class_weight='balanced')
clf.fit(X_train, train_df['stance'])
preds = clf.predict(X_test)

print("Test Macro F1:", f1_score(test_df['stance'], preds, average='macro'))
print(classification_report(test_df['stance'], preds, digits=4))

### 5. Transformer Fine-Tuning: Google MuRIL (Multilingual Representations for Indian Languages)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

MODEL_NAME = 'google/muril-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {'Against': 0, 'Favour': 1, 'None': 2}
id2label = {0: 'Against', 1: 'Favour', 2: 'None'}

def tokenize_fn(batch):
    return tokenizer(batch['cleaned'], padding='max_length', truncation=True, max_length=128)

train_ds = Dataset.from_pandas(train_df[['cleaned', 'stance']].assign(label=train_df['stance'].map(label2id)))
val_ds = Dataset.from_pandas(val_df[['cleaned', 'stance']].assign(label=val_df['stance'].map(label2id)))
test_ds = Dataset.from_pandas(test_df[['cleaned', 'stance']].assign(label=test_df['stance'].map(label2id)))

train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok = val_ds.map(tokenize_fn, batched=True)
test_tok = test_ds.map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

training_args = TrainingArguments(
    output_dir='./muril_checkpoints',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1'
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    macro_f1 = f1_score(labels, preds, average='macro')
    return {'macro_f1': macro_f1}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics
)

# Run fine-tuning on GPU
# trainer.train()

### 6. Generate Submission Predictions

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'predicted_label': preds
})
submission.to_csv('test_predictions.csv', index=False)
print("Submission saved to test_predictions.csv")
submission.head(10)